###  Text Preprocessing Pipeline
A complete preprocessing system that takes raw text input and outputs clean, structured text ready for machine learning models.
This pipeline should handle:
1. Cleaning raw text
2. Custom tokenization (not just default library use)
3. Stopword removal
4. Lemmatization
5. Output formatted processed text


In [34]:
# import the required libraries for this project
# note: we will be using Spacy instead of NLTK beacuse of its efficiency, speed and ease of use for NLP tasks

import spacy
import re
import pandas as pd
import numpy as np
import warnings
import logging
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Load spaCy model for lemmatization
nlp = spacy.load("en_core_web_sm")

# English stopwords list
stop_words = set(stopwords.words('english'))

#remove warnings
import warnings
warnings.filterwarnings('ignore')

In [35]:
'''Logging alllows us to:
1.Track pipeline execution step-by-step
2.Debug errors in real systems
3.Monitor data flow in production
4.Store history of what happened internally'''

logging.basicConfig(
    filename="nlp_pipeline.log",   # log file
    filemode='a',           # append mode
    level=logging.INFO,        # log level
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

In [36]:
# contractions dictionary, we will use this to expand contractions in the text, like "don't" to "do not", "can't" to "cannot"
contractions_dict = {
    "I'm": "I am",
    "I'm'a": "I am about to",
    "I'm'o": "I am going to",
    "I've": "I have",
    "I'll": "I will",
    "I'll've": "I will have",
    "I'd": "I would",
    "I'd've": "I would have",
    "Whatcha": "What are you",
    "amn't": "am not",
    "ain't": "are not",
    "aren't": "are not",
    "'cause": "because",
    "can't": "cannot",
    "can't've": "cannot have",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "daren't": "dare not",
    "daresn't": "dare not",
    "dasn't": "dare not",
    "didn't": "did not",
    "didn’t": "did not",
    "don't": "do not",
    "don’t": "do not",
    "doesn't": "does not",
    "e'er": "ever",
    "everyone's": "everyone is",
    "finna": "fixing to",
    "gimme": "give me",
    "gon't": "go not",
    "gonna": "going to",
    "gotta": "got to",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he've": "he have",
    "he's": "he is",
    "he'll": "he will",
    "he'll've": "he will have",
    "he'd": "he would",
    "he'd've": "he would have",
    "here's": "here is",
    "how're": "how are",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how's": "how is",
    "how'll": "how will",
    "isn't": "is not",
    "it's": "it is",
    "'tis": "it is",
    "'twas": "it was",
    "it'll": "it will",
    "it'll've": "it will have",
    "it'd": "it would",
    "it'd've": "it would have",
    "kinda": "kind of",
    "let's": "let us",
    "luv": "love",
    "ma'am": "madam",
    "may've": "may have",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "ne'er": "never",
    "o'": "of",
    "o'clock": "of the clock",
    "ol'": "old",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "o'er": "over",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shalln't": "shall not",
    "shan't've": "shall not have",
    "she's": "she is",
    "she'll": "she will",
    "she'd": "she would",
    "she'd've": "she would have",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "somebody's": "somebody is",
    "someone's": "someone is",
    "something's": "something is",
    "sux": "sucks",
    "that're": "that are",
    "that's": "that is",
    "that'll": "that will",
    "that'd": "that would",
    "that'd've": "that would have",
    "'em": "them",
    "there're": "there are",
    "there's": "there is",
    "there'll": "there will",
    "there'd": "there would",
    "there'd've": "there would have",
    "these're": "these are",
    "they're": "they are",
    "they've": "they have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they'd": "they would",
    "they'd've": "they would have",
    "this's": "this is",
    "this'll": "this will",
    "this'd": "this would",
    "those're": "those are",
    "to've": "to have",
    "wanna": "want to",
    "wasn't": "was not",
    "we're": "we are",
    "we've": "we have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we'd": "we would",
    "we'd've": "we would have",
    "weren't": "were not",
    "what're": "what are",
    "what'd": "what did",
    "what've": "what have",
    "what's": "what is",
    "what'll": "what will",
    "what'll've": "what will have",
    "when've": "when have",
    "when's": "when is",
    "where're": "where are",
    "where'd": "where did",
    "where've": "where have",
    "where's": "where is",
    "which's": "which is",
    "who're": "who are",
    "who've": "who have",
    "who's": "who is",
    "who'll": "who will",
    "who'll've": "who will have",
    "who'd": "who would",
    "who'd've": "who would have",
    "why're": "why are",
    "why'd": "why did",
    "why've": "why have",
    "why's": "why is",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "you're": "you are",
    "you've": "you have",
    "you'll've": "you shall have",
    "you'll": "you will",
    "you'd": "you would",
    "you'd've": "you would have",

    "to cause": "to cause",
    "will cause": "will cause",
    "should cause": "should cause",
    "would cause": "would cause",
    "can cause": "can cause",
    "could cause": "could cause",
    "must cause": "must cause",
    "might cause": "might cause",
    "shall cause": "shall cause",
    "may cause": "may cause"
}

In [37]:
def validate_input(text):
    """
    STEP 0: Ensures safe pipeline execution
    MUST BE USED BEFORE ANY PROCESSING
    """

    if text is None:
        return False
    if not isinstance(text, str):
        return False
    if text.strip() == "":
        return False

    return True

In [38]:
# the first step is to remove contractions, like "don't" to "do not", "can't" to "cannot"
def expand_contractions(text):
    """
    Expands contractions like don't → do not
    """

    if not isinstance(text, str):
        return ""

    pattern = re.compile(r'\b(' + '|'.join(contractions_dict.keys()) + r')\b')

    return pattern.sub(lambda x: contractions_dict[x.group()], text)

In [39]:
# step 1: clean the text data by removing special characters, numbers, punctuation, and lowercasing the text
def clean_text(text):
    """
    Cleans raw text:
    - lowercase
    - remove URLs
    - remove emails
    - remove special characters
    - normalize spaces
    """

    if not isinstance(text, str) or text.strip() == "":
        return ""

    text = text.lower()

    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove emails
    text = re.sub(r'\S+@\S+', '', text)

    # keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [40]:
# Step 2: creating a coustom tokenizer using regex to split the text into tokens (words)
def custom_tokenizer(text):
    """
    ADVANCED CUSTOM TOKENIZER (IMPROVED)
    - Handles alphanumeric tokens
    - Splits intelligently
    - Removes noise
    """

    raw_tokens = re.findall(r'\b[a-zA-Z0-9]+\b', text)

    tokens = []

    for token in raw_tokens:
        token = token.lower()

        # remove numbers inside words (ai2024 → ai)
        token = re.sub(r'\d+', '', token)

        # skip empty after cleaning
        if token == "":
            continue

        # remove repeated chars (coooool → cool)
        token = re.sub(r'(.)\1{2,}', r'\1\1', token)

        # remove useless tokens
        if len(token) < 2:
            continue

        if token in ["thing", "something", "anything", "everything"]:
            continue

        tokens.append(token)

    return tokens

In [41]:
# we will remove punctuation here, its best here because it can creates problems later on real data.
def remove_stopwords(tokens):
    """
    Removes stopwords + noise filtering
    """

    filtered = []

    for t in tokens:
        t = t.lower().strip()

        if t in stop_words:
            continue

        if len(t) < 2:
            continue

        filtered.append(t)

    return filtered

In [42]:
# step 4: lemmatization using spacy
def lemmatize_text(tokens):

    text = " ".join(tokens)

    doc = nlp(text, disable=["parser", "ner"])

    return [
        token.lemma_
        for token in doc
        if token.lemma_ != "-PRON-"
    ]

In [43]:
# Integrating all the steps into a single function to create a text processing pipeline
def text_preprocessing_pipeline(text, config=None):
    """
    FINAL INDUSTRY-LEVEL TEXT PREPROCESSING PIPELINE
    - Safe
    - Optimized
    - Production-ready
    """

    if config is None:
        config = {
            "remove_stopwords": True,
            "lemmatize": True,
            "remove_duplicates": True
        }

    try:
        logger.info("Pipeline started")

        # ✅ Step 0: Validation
        if not validate_input(text):
            logger.warning("Invalid input received")
            return []

        logger.info(f"Raw Input: {text}")

        # ✅ Step 1: Expand contractions (SAFE VERSION)
        pattern = re.compile(
            r'\b(' + '|'.join(map(re.escape, contractions_dict.keys())) + r')\b'
        )
        text = pattern.sub(lambda x: contractions_dict[x.group()], text)
        logger.info(f"After contraction expansion: {text}")

        # ✅ Step 2: Cleaning
        text = text.lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()

        logger.info(f"After cleaning: {text}")

        # ✅ Step 3: Tokenization (optimized)
        raw_tokens = re.findall(r'\b[a-zA-Z]+\b', text)

        tokens = []
        for token in raw_tokens:
            token = token.lower()

            # remove repeated chars (coooool → cool)
            token = re.sub(r'(.)\1{2,}', r'\1\1', token)

            if len(token) < 2:
                continue

            if token in {"thing", "something", "anything", "everything"}:
                continue

            tokens.append(token)

        logger.info(f"Tokens: {tokens}")

        # ✅ Step 4: Stopword removal
        if config.get("remove_stopwords", True):
            tokens = [t for t in tokens if t not in stop_words]
            logger.info(f"After stopword removal: {tokens}")

        # ✅ Step 5: Lemmatization (FAST MODE)
        if config.get("lemmatize", True) and tokens:
            doc = nlp(" ".join(tokens), disable=["parser", "ner"])
            tokens = [
                token.lemma_
                for token in doc
                if token.lemma_ != "-PRON-"
            ]
            logger.info(f"After lemmatization: {tokens}")

        # ✅ Step 6: Remove duplicates (order preserved)
        if config.get("remove_duplicates", True):
            tokens = list(dict.fromkeys(tokens))
            logger.info(f"After duplicate removal: {tokens}")

        logger.info(f"Final Output: {tokens}")

        return tokens

    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        return []

In [44]:
# comparing stemming vs lemmatization using your pipeline
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def compare_stem_vs_lemma(text):

    if not validate_input(text):
        return None

    print("\nRAW:", text)

    lemma = text_preprocessing_pipeline(text)

    clean = clean_text(expand_contractions(text))
    tokens = custom_tokenizer(clean)
    tokens = remove_stopwords(tokens)

    stemmed = [stemmer.stem(t) for t in tokens]

    print("\nSTEMMED:", stemmed)
    print("\nLEMMATIZED:", lemma)

    return {"stemmed": stemmed, "lemmatized": lemma}

In [45]:
# batch processing of multiple texts using the pipeline and also to get data from csv files.
def preprocess_batch(data, text_column=None, method="lemma"):

    import pandas as pd
    from nltk.stem import PorterStemmer

    stemmer = PorterStemmer()

    # -----------------------
    # Load / extract texts
    # -----------------------
    if isinstance(data, str):
        data = pd.read_csv(data)

    if isinstance(data, pd.DataFrame):
        if text_column is None:
            raise ValueError("text_column must be provided for DataFrame input")
        texts = data[text_column].fillna("").tolist()

    elif isinstance(data, list):
        texts = data

    else:
        raise ValueError("Unsupported input type")

    # -----------------------
    # Processing
    # -----------------------
    results = []

    for text in texts:

        if not validate_input(text):
            results.append("")
            continue

        # 🔥 STEM MODE
        if method == "stem":
            text_clean = clean_text(expand_contractions(text))
            tokens = custom_tokenizer(text_clean)
            tokens = remove_stopwords(tokens)
            processed = [stemmer.stem(t) for t in tokens]

        # 🔥 LEMMA MODE (DEFAULT)
        else:
            processed = text_preprocessing_pipeline(text)

        results.append(" ".join(processed))

    return results

In [46]:
sample_text = "I've been running, trying, and playing with better tools!!!"

text_preprocessing_pipeline(sample_text)

['run', 'try', 'play', 'well', 'tool']

In [47]:
df = pd.read_csv("dataset.csv")

df["processed_text"] = preprocess_batch(df, "text")

In [48]:
def compare_raw_vs_processed(text):

    if not validate_input(text):
        print("Invalid input")
        return []

    print("\nRAW:", text)

    processed = text_preprocessing_pipeline(text)

    print("\nPROCESSED:", processed)

    return processed

In [49]:
def run_cli():
    while True:
        print("\n===== NLP PIPELINE =====")
        print("1. Single Text")
        print("2. Raw vs Processed")
        print("3. Stem vs Lemma")
        print("4. Batch CSV")
        print("5. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            text = input("Enter text: ")
            print(text_preprocessing_pipeline(text))

        elif choice == "2":
            text = input("Enter text: ")
            compare_raw_vs_processed(text)

        elif choice == "3":
            text = input("Enter text: ")
            compare_stem_vs_lemma(text)

        elif choice == "4":
            path = input("CSV path: ")
            col = input("Column name: ")

            df = pd.read_csv(path)
            df["processed_text"] = preprocess_batch(df, col)

            df.to_csv("processed_output.csv", index=False)
            print("Saved as processed_output.csv")

        elif choice == "5":
            break

        else:
            print("Invalid choice")